In [1]:
import json
import ast

# system message
system_msg = 'You are a helpful assistant who is an expert in Kubernetes troubleshooting and can help developers with step by step suggestions about how to investigate and solve Kubernetes problems.'

# File paths
input_file_path = './data/raw_data.txt'
output_file_path = './data/kub_data_mrc.jsonl'

# Read the input file
with open(input_file_path, 'r') as infile:
    lines = infile.readlines()

# Process the file to extract prompt-completion pairs
mrc_dicts = []

for line in lines:
    splitted = line.split('\t```json')
    user_msg = splitted[0]
    # assistant_msg = json.loads(splitted[1])
    # assistant_msg = ast.literal_eval(splitted[1])
    assistant_msg = splitted[1]
    mrc_dicts.append({"messages" : [{"role": "system", "content": system_msg}, 
                         {"role": "user", "content": user_msg}, 
                         {"role": "assistant", "content": assistant_msg}]})    

with open(output_file_path, 'w') as outfile:
    for dct in mrc_dicts:
        json.dump(dct, outfile)
        outfile.write('\n')

mrc_file = output_file_path

In [2]:
from openai import OpenAI
import warnings
import os

In [3]:
os.environ['OPENAI_API_KEY'] = ""
client = OpenAI()

In [4]:
# Upload the file
response_file = client.files.create(
    file=open(mrc_file, "rb"),
    purpose="fine-tune"
)

In [5]:
# Create the fine-tune job
response_fine_tune = client.fine_tuning.jobs.create(
    training_file=response_file.id,
    model='gpt-3.5-turbo-0125',
    suffix='fahim'
)

In [6]:
model_name = 'ft:gpt-3.5-turbo-0125:personal:fahim:9uG09DCK'

def ask_model_mrc(question):
    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": question}
        ]
    )
    return completion.choices[0].message.content

In [7]:
question = "Kubernetes - Failed Persistent Volume Claims"
ask_model_mrc(question)

'{  "investigation": {    "steps": [      {        "description": "List Persistent Volume Claims (PVCs)",        "command": "kubectl get pvc"      },      {        "description": "Check status of a specific PVC in more detail",        "command": "kubectl describe pvc [PVC_NAME]"      },      {        "description": "Check for events related to the PVC",        "command": "kubectl describe events --field-selector involvedObject.name=[PVC_NAME]"      },      {        "description": "Check if the Persistent Volume (PV) is properly bound to the PVC",        "command": "kubectl get pv --sort-by=.status.phase; kubectl get pvc | grep -i unbound"      },      {        "description": "Check storage class definition and status",        "command": "kubectl get storageclass; kubectl describe storageclass [STORAGE_CLASS_NAME]"      },      {        "description": "Check if there are enough available storage resources for pod/pvc creation",        "command": "kubectl describe storageclass [STORAGE_C

In [53]:
import gradio as gr

In [65]:
# Initialize a list to store conversation history.
conversation_history_master = []
conversation_history_ft = []
conversation_history_s = []

In [66]:
def chat_with_openai(message):
    global conversation_history_master
    global conversation_history_ft
    global conversation_history_s
    conversation_history_master.append({"role": "user", "content": message})
    conversation_history_ft.append({"role": "user", "content": message})
    conversation_history_s.append({"role": "user", "content": message})

    try:
        # Make the API call to the fine-tuned model with the full conversation history.
        completion_ft = client.chat.completions.create(
            model=model_name,
            messages=conversation_history_ft
        )

        completion_s = client.chat.completions.create(
            model="gpt-4",
            messages=conversation_history_s
        )

        # Extract the response and append it to the conversation history.
        answer_finetuned = completion_ft.choices[0].message.content
        answer_stock = completion_s.choices[0].message.content
        conversation_history_master.append({"role": "assistant", "content": answer_finetuned})
        conversation_history_master.append({"role": "assistant", "content": answer_stock})
        conversation_history_ft.append({"role": "assistant", "content": answer_finetuned})
        conversation_history_s.append({"role": "assistant", "content": answer_stock})
    except Exception as e:
        return f"Error: {str(e)}"
    # print(conversation_history_master)
    return answer_finetuned

In [67]:
def respond(message):
    _ = chat_with_openai(message)
    # Convert the conversation history to a format suitable for gr.Chatbot
    chat_history = [(entry['content'], "") if entry['role'] == 'user' else ("", entry['content']) for entry in conversation_history_master]
    return "", chat_history

In [68]:
with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    msg.submit(respond, inputs=msg, outputs=[msg, chatbot])
demo.launch()

Running on local URL:  http://127.0.0.1:7875

To create a public link, set `share=True` in `launch()`.
